# Advanced: hybrid retrieval, fusion and reranking

Hybrid retrieval combines lexical precision with semantic recall. This self-contained experiment uses word and character TF-IDF as two complementary channels, fuses them with RRF and evaluates each stage against query-level relevance judgments.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nlp_portfolio.rag import reciprocal_rank_fusion
from nlp_portfolio.retrieval_eval import evaluate_run

documents = {
    "d1": "BRCA1 participates in homologous recombination repair of DNA double-strand breaks.",
    "d2": "EGFR inhibitors are used in tumors carrying sensitizing EGFR mutations.",
    "d3": "TP53 encodes a tumor suppressor that responds to DNA damage.",
    "d4": "Homology-directed repair uses a homologous DNA template.",
    "d5": "The epidermal growth factor receptor activates MAPK signaling.",
}
queries = {
    "q1": "Which gene is involved in homology based repair?",
    "q2": "What receptor is targeted in mutation-positive cancer?",
    "q3": "Which tumor suppressor reacts to genome damage?",
}
qrels = {"q1": {"d1", "d4"}, "q2": {"d2", "d5"}, "q3": {"d3"}}

In [ ]:
def tfidf_run(analyzer, ngram_range):
    ids = list(documents)
    vectorizer = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram_range, sublinear_tf=True)
    matrix = vectorizer.fit_transform(documents.values())
    run = {}
    for query_id, text in queries.items():
        scores = cosine_similarity(vectorizer.transform([text]), matrix)[0]
        run[query_id] = [ids[i] for i in np.argsort(-scores)]
    return run

word_run = tfidf_run("word", (1, 2))
character_run = tfidf_run("char_wb", (3, 5))
fused_run = {query_id: reciprocal_rank_fusion([word_run[query_id], character_run[query_id]])
             for query_id in queries}

pd.DataFrame({
    "word": evaluate_run(word_run, qrels, k=3),
    "character": evaluate_run(character_run, qrels, k=3),
    "RRF": evaluate_run(fused_run, qrels, k=3),
}).round(3)

Character n-grams help with morphology, spelling variation and biomedical symbols. Word n-grams preserve phrase specificity. Fusion can improve robustness, although gains must be verified over many queries.

## Optional cross-encoder reranking

A cross-encoder jointly reads the query and passage. It is usually more accurate and more expensive than independent embedding retrieval, so it reranks only a short candidate list.

In [ ]:
RUN_RERANKER = False
reranked_run = {}
if RUN_RERANKER:
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    for query_id, query in queries.items():
        candidates = fused_run[query_id][:4]
        scores = reranker.predict([(query, documents[item]) for item in candidates])
        reranked_run[query_id] = [candidates[i] for i in np.argsort(-scores)]
    print(evaluate_run(reranked_run, qrels, k=3))

## Retrieval trace and failure localization

Logging each stage makes it possible to see whether a relevant document was lost during candidate generation or reordered by the reranker.

In [ ]:
trace = []
for query_id in queries:
    trace.append({
        "query": query_id,
        "relevant": sorted(qrels[query_id]),
        "word_top3": word_run[query_id][:3],
        "character_top3": character_run[query_id][:3],
        "fused_top3": fused_run[query_id][:3],
    })
pd.DataFrame(trace)

For a real RAG system, annotate relevance independently, tune candidate depth on validation queries and reserve a test set. Report both retrieval metrics and grounded answer metrics. Generation quality cannot compensate when evidence is absent from the retrieved context.